# Transformation step verification

Runs the two pipeline steps in order:
1) transform catalog OntoUML JSON to Alloy files
2) verify generated Alloy models for parse + executability (batch run).

In [5]:
import pathlib

version_path = pathlib.Path("../../src/main/resources/ontouml2alloy/VERSION")
print("OntoUML2Alloy version (git commit hash):", version_path.read_text().strip())

OntoUML2Alloy version (git commit hash): abccfe95f76419c3554627e169470f86cdb633c9


## 1) Transform catalog JSON to Alloy batch output

Uses the crawl manifest as input and writes generated Alloy files to `evaluation/transformation-verification/results/batch-alloy-output`.

In [6]:
%%bash
set -e

cd ../../

OUT_DIR="evaluation/transformation-verification/results/batch-alloy-output"
rm -rf "$OUT_DIR"

mvn compile -q exec:java \
  -Dexec.mainClass="com.example.batch.CatalogTransformationRunner" \
  -Dexec.args="evaluation/catalog-download/ontouml_downloads/manifest.json $OUT_DIR"

144 models
[1/144] university-ontology ... OK
[2/144] gailly2016value ... FAILED: ontouml2alloy-transformation exited with code 1
[3/144] franco2018rpg ... FAILED: ontouml2alloy-transformation exited with code 1
[4/144] chartered-service ... OK
[5/144] saleme2019mulseonto ... FAILED: ontouml2alloy-transformation exited with code 1
[6/144] sousa2022triponto ... OK
[7/144] buridan-ontology2021 ... OK
[8/144] sales2018competition ... OK
[9/144] elghosh2020cargos ... OK
[10/144] stock-broker2021 ... OK
[11/144] lindeberg2022full-ontorights ... FAILED: ontouml2alloy-transformation exited with code 1
[12/144] spmo2017 ... FAILED: ontouml2alloy-transformation exited with code 1
[13/144] project-management-ontology ... OK
[14/144] sportbooking2021 ... OK
[15/144] construction-model ... OK
[16/144] lindeberg2022simple-ontorights ... OK
[17/144] fumagalli2022criminal-investigation ... OK
[18/144] derave2019dpo ... FAILED: ontouml2alloy-transformation exited with code 1
[19/144] duarte2018osdef .

## 2) Verify generated Alloy models (parse + executability)

Reads the transform manifest from `evaluation/transformation-verification/results/batch-alloy-output` and writes verification JSON to `evaluation/transformation-verification/results/verification`.

Optional **slug** exclusions: set `EXCLUDE_SLUGS_CSV` in the next cell (comma-separated catalog slugs; slugs do not contain commas).

In [7]:
%%bash
set -e

# Optional comma-separated catalog slugs to skip
# EXCLUDE_SLUGS="barcelos2015transport-networks,mgic-antt2011"
EXCLUDE_SLUGS=""

cd ../../

mvn clean -q compile exec:java \
  -Dexec.mainClass="com.example.batch.CatalogAlloyRunner" \
  -Dexec.args="evaluation/transformation-verification/results/batch-alloy-output/alloy_batch_manifest.json evaluation/transformation-verification/results/verification/verification_results.json ${EXCLUDE_SLUGS}"

Running 100 models

[1/100] university-ontology/main.als : 

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


SAT (1520ms)
[2/100] chartered-service/main.als : SAT (116ms)
[3/100] sousa2022triponto/main.als : SAT (4016ms)
[4/100] buridan-ontology2021/main.als : SAT (5039ms)
[5/100] sales2018competition/main.als : UNSAT (1213ms)
[6/100] elghosh2020cargos/main.als : SAT (39ms)
[7/100] stock-broker2021/main.als : SAT (190ms)
[8/100] project-management-ontology/main.als : SAT (193ms)
[9/100] sportbooking2021/main.als : SAT (3760ms)
[10/100] construction-model/main.als : SAT (179ms)
[11/100] lindeberg2022simple-ontorights/main.als : ERROR (134ms)
[12/100] fumagalli2022criminal-investigation/main.als : SAT (209ms)
[13/100] khantong2020ontology/main.als : SAT (726ms)
[14/100] silva2012itarchitecture/main.als : SAT (8011ms)
[15/100] porello2020coex/main.als : SAT (194ms)
[16/100] tourbo2021/main.als : UNSAT (4488ms)
[17/100] castro2012cloudvulnerability/main.als : SAT (1217ms)
[18/100] social-contract/main.als : UNSAT (853ms)
[19/100] zhou2017hazard-ontology-train-control/main.als : SAT (54ms)
[20/100

## 3) Compare original vs transformed class counts

Counts classes from each source `ontology.json` and compares them to `transformation_metadata.json` to estimate:
- original classes
- transformed classes
- removed classes
- keep ratio

In [8]:
from pathlib import Path
import json
import statistics

orig_root = Path("../catalog-download/ontouml_downloads")
out_root = Path("results/batch-alloy-output")

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8", errors="replace"))

def walk(node):
    if isinstance(node, dict):
        yield node
        for value in node.values():
            yield from walk(value)
    elif isinstance(node, list):
        for item in node:
            yield from walk(item)

def class_ids_from_ontology(ontology_json: dict):
    ids = set()
    for n in walk(ontology_json):
        if n.get("type") != "Class":
            continue
        class_id = n.get("id")
        if class_id:
            ids.add(class_id)
    return ids

def class_ids_from_metadata(metadata_json: dict):
    ids = set()
    for c in metadata_json.get("classes", []):
        class_id = c.get("id")
        if class_id:
            ids.add(class_id)
    return ids


rows = []

for meta_path in sorted(out_root.glob("*/transformation_metadata.json")):
    slug = meta_path.parent.name
    src_path = orig_root / slug / "ontology.json"
    source_ids = class_ids_from_ontology(load_json(src_path))
    transformed_ids = class_ids_from_metadata(load_json(meta_path))
    removed_ids = source_ids - transformed_ids
    rows.append(
        {
            "slug": slug,
            "original_classes": len(source_ids),
            "transformed_classes": len(transformed_ids),
            "removed_classes": len(removed_ids),
            "keep_ratio_pct": round(len(transformed_ids) / len(source_ids) * 100.0, 1),
        }
    )

total_original = sum(r["original_classes"] for r in rows)
total_transformed = sum(r["transformed_classes"] for r in rows)

none_removed_count = sum(1 for r in rows if r["removed_classes"] == 0)
worst_case = max(rows, key=lambda r: r["removed_classes"])

print(f"Overall keep ratio: {(total_transformed / total_original)*100:.1f}%")
print(f"Median removed classes per model: {statistics.median(r['removed_classes'] for r in rows)}")
print(f"Models with none removed: {none_removed_count}")
print(f"Worst case: {worst_case['slug']}, removed={worst_case['removed_classes']/worst_case['original_classes']*100:.1f}%")

with_removed = [r["keep_ratio_pct"] for r in rows if r["removed_classes"] > 0]
print(f"Mean keep ratio for models with removed classes: {statistics.mean(with_removed):.1f}% ({len(with_removed)} models)")

Overall keep ratio: 86.0%
Median removed classes per model: 0.0
Models with none removed: 51
Worst case: mgic-antt2011, removed=20.2%
Mean keep ratio for models with removed classes: 77.9% (49 models)


In [12]:
import json
import pandas as pd

with open("results/verification/verification_results.json") as f:
    update_data = json.load(f)

with open("results-baseline/verification/verification_results.json") as f:
    baseline_data = json.load(f)


def result_label(entry):
    if "error" in entry:
        return "Error"
    return "SAT" if entry.get("satisfiable") else "UNSAT"


update_map = {e["slug"]: e for e in update_data["results"]}
baseline_map = {e["slug"]: e for e in baseline_data["results"]}

all_slugs = sorted(set(update_map) | set(baseline_map))

rows = []
for slug in all_slugs:
    u = update_map.get(slug)
    b = baseline_map.get(slug)
    rows.append({
        "Model": u["modelName"] if u else b["modelName"],
        "Baseline": result_label(b) if b else "—",
        "Update": result_label(u) if u else "—",
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
df

,Model,Baseline,Update
0,Petroleum System Model,SAT,SAT
1,Relational Database System Ontology,Error,SAT
2,Reference Ontology on Object-Oriented Code,Error,Error
3,Aviation Safety Ontology,Error,—
4,OntoBio,—,SAT
5,AlpineBits DestinationData Ontology,—,SAT
6,Reference Ontology of Trust,—,SAT
7,Game Theory Ontology,Error,SAT
8,Reference Ontology of Money and Virtual Curren...,—,UNSAT
9,Reference Ontology of Ethical Requirements,SAT,SAT
